# McMath Solar Telescope — 핵심 광학 계산 구현 / Core Optical Calculations Implementation

**논문 / Paper**: A. Keith Pierce, "The McMath Solar Telescope of Kitt Peak National Observatory," *Applied Optics*, Vol. 3, No. 12, 1964.

이 노트북은 McMath Solar Telescope 논문의 핵심 광학·기기 설계 원리를 계산과 시각화를 통해 재현합니다.

This notebook reproduces the core optical and instrumental design principles of the McMath Solar Telescope paper through calculations and visualizations.

**구성 / Contents:**
1. 이미지 스케일과 태양상 크기 / Image Scale and Solar Image Size
2. 회절 한계 vs. 실제 Seeing / Diffraction Limit vs. Actual Seeing
3. 분광기 설계의 역방향 논리 / Backward Logic of Spectrograph Design
4. 격자 분해능과 분산 / Grating Resolving Power and Dispersion
5. 열 변형 계수 비교: Quartz vs. Metal / Thermal Distortion Factor: Quartz vs. Metal
6. Double-Pass 산란광 감소 효과 / Double-Pass Scattered Light Reduction
7. S/N Ratio: Image Slicer 유무에 따른 비교 / S/N with and without Image Slicer
8. McMath 광학 경로 시각화 / McMath Optical Path Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: 이미지 스케일과 태양상 크기 / Image Scale and Solar Image Size

초점면에서의 이미지 스케일은 초점 거리 $f$에 의해 결정됩니다:

The image scale at the focal plane is determined by the focal length $f$:

$$s = \frac{f}{206265} \quad \text{(mm/arcsec)}$$

여기서 206,265는 1 radian의 arcsec 수입니다. McMath는 90m 초점 거리로 태양 전면(~32 arcmin) 이미지가 약 84cm — "approximately a yard across"가 됩니다.

where 206,265 is the number of arcseconds per radian. McMath's 90-m focal length produces a full solar disk (~32 arcmin) image of ~84 cm — "approximately a yard across."

In [ ]:
def image_scale(focal_length_m):
    """Compute image scale in mm/arcsec given focal length in meters."""
    return (focal_length_m * 1000) / 206265


def solar_image_diameter(focal_length_m, solar_angular_diam_arcmin=32.0):
    """Compute solar disk image diameter in mm."""
    scale = image_scale(focal_length_m)  # mm/arcsec
    return scale * solar_angular_diam_arcmin * 60  # arcsec -> mm


# McMath parameters
f_mcmath = 90.0  # meters
scale_mcmath = image_scale(f_mcmath)
solar_diam_mcmath = solar_image_diameter(f_mcmath)

print("=" * 60)
print("McMath Solar Telescope — Image Scale")
print("=" * 60)
print(f"Focal length:           {f_mcmath:.0f} m")
print(f"Image scale:            {scale_mcmath:.4f} mm/arcsec")
print(f"Solar image diameter:   {solar_diam_mcmath:.1f} mm = {solar_diam_mcmath/10:.1f} cm")
print(f"1 arcsec granule size:  {scale_mcmath:.3f} mm = {scale_mcmath*1000:.1f} μm")
print()

# Compare with other solar telescopes
telescopes = {
    "McMath-Pierce (1964)": 90.0,
    "Dunn Solar (1969)": 54.9,
    "Swedish Solar (2002)": 20.3,
    "DKIST (2020)": 8.0 * 18,  # effective with reimaging
    "SDO/AIA (2010)": 4.125,
}

print(f"{'Telescope':<30} {'f (m)':>8} {'Scale (mm/\")':>14} {'Solar Ø (cm)':>14}")
print("-" * 70)
for name, f in telescopes.items():
    s = image_scale(f)
    d = solar_image_diameter(f) / 10  # mm -> cm
    print(f"{name:<30} {f:>8.1f} {s:>14.4f} {d:>14.1f}")

In [ ]:
# Visualize: image scale vs focal length, with solar features marked
focal_lengths = np.linspace(1, 150, 300)
scales = image_scale(focal_lengths)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: image scale vs focal length
ax1.plot(focal_lengths, scales, "b-", linewidth=2)
ax1.axhline(y=scale_mcmath, color="r", linestyle="--", alpha=0.7, label=f"McMath: {scale_mcmath:.3f} mm/\"")
ax1.axvline(x=f_mcmath, color="r", linestyle="--", alpha=0.3)

# Mark other telescopes
markers = {"Dunn": 54.9, "SST": 20.3, "AIA": 4.125}
for name, f in markers.items():
    s = image_scale(f)
    ax1.plot(f, s, "ko", markersize=8)
    ax1.annotate(name, (f, s), textcoords="offset points", xytext=(5, 10), fontsize=10)
ax1.plot(f_mcmath, scale_mcmath, "r*", markersize=15)
ax1.annotate("McMath", (f_mcmath, scale_mcmath), textcoords="offset points",
             xytext=(5, 10), fontsize=10, color="r", fontweight="bold")

ax1.set_xlabel("Focal Length (m)")
ax1.set_ylabel("Image Scale (mm/arcsec)")
ax1.set_title("Image Scale vs Focal Length\n이미지 스케일 vs 초점 거리")
ax1.legend()

# Right: solar image diameter vs focal length
diameters = solar_image_diameter(focal_lengths) / 10  # cm
ax2.plot(focal_lengths, diameters, "orange", linewidth=2)
ax2.axhline(y=solar_diam_mcmath / 10, color="r", linestyle="--", alpha=0.7,
            label=f"McMath: {solar_diam_mcmath/10:.1f} cm")

# Mark 1 yard ≈ 91.4 cm
ax2.axhline(y=91.4, color="green", linestyle=":", alpha=0.5, label='1 yard ≈ 91.4 cm')

ax2.plot(f_mcmath, solar_diam_mcmath / 10, "r*", markersize=15)

# Show key solar feature sizes at McMath scale
features = {"Granule (~1\")": 1, "Sunspot umbra (~10\")": 10, "Active region (~100\")": 100}
for name, arcsec in features.items():
    size_mm = scale_mcmath * arcsec
    ax2.annotate(f"{name}\n→ {size_mm:.1f} mm at McMath",
                 xy=(130, solar_diam_mcmath / 10 * 0.3 + arcsec * 0.3),
                 fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

ax2.set_xlabel("Focal Length (m)")
ax2.set_ylabel("Solar Image Diameter (cm)")
ax2.set_title("Solar Disk Image Size vs Focal Length\n태양상 크기 vs 초점 거리")
ax2.legend()

plt.tight_layout()
plt.show()

## Part 2: 회절 한계 vs. 실제 Seeing / Diffraction Limit vs. Actual Seeing

회절 한계 분해능은 구경 $D$에 의해 결정됩니다:

The diffraction-limited resolution is determined by the aperture $D$:

$$\theta_{\text{diff}} = 1.22 \frac{\lambda}{D} \quad \text{(radians)} = \frac{1.22 \lambda}{D} \times 206265 \quad \text{(arcsec)}$$

Pierce는 McMath의 설계 목표로 0.33 arcsec를 언급했으나, 실제 주간 seeing은 ~1.5 arcsec (Livingston 측정, nearly Gaussian)이었습니다. 이 간극이 왜 단순히 구경을 키우는 것만으로는 부족한지를 보여줍니다.

Pierce mentions 0.33 arcsec as the design goal, but actual daytime seeing was ~1.5 arcsec (Livingston measurement). This gap shows why simply increasing aperture is insufficient.

In [ ]:
def diffraction_limit_arcsec(wavelength_nm, aperture_cm):
    """Compute diffraction-limited resolution in arcsec."""
    lam = wavelength_nm * 1e-7  # nm -> cm
    return 1.22 * lam / aperture_cm * 206265


# Aperture range
apertures_cm = np.linspace(5, 400, 500)
diff_limit_550 = diffraction_limit_arcsec(550, apertures_cm)

fig, ax = plt.subplots(figsize=(12, 7))

# Diffraction limit curve
ax.plot(apertures_cm, diff_limit_550, "b-", linewidth=2.5, label="Diffraction limit (λ=550 nm)")

# Typical seeing conditions
seeing_levels = {
    "Excellent daytime seeing": (0.5, "green"),
    "McMath typical daytime (~1.5\")": (1.5, "orange"),
    "Median daytime seeing (~2\")": (2.0, "red"),
}
for label, (val, color) in seeing_levels.items():
    ax.axhline(y=val, color=color, linestyle="--", alpha=0.6, linewidth=1.5, label=label)

# Mark telescopes
tel_data = {
    "McMath\n(160 cm)": (160, "r"),
    "Dunn\n(76 cm)": (76, "purple"),
    "SST\n(100 cm)": (100, "darkgreen"),
    "DKIST\n(400 cm)": (400, "navy"),
}
for name, (apt, color) in tel_data.items():
    dl = diffraction_limit_arcsec(550, apt)
    ax.plot(apt, dl, "o", color=color, markersize=10, zorder=5)
    ax.annotate(name, (apt, dl), textcoords="offset points", xytext=(8, 8),
                fontsize=10, color=color, fontweight="bold")

# Design goal from paper
ax.axhline(y=0.33, color="gray", linestyle=":", alpha=0.5)
ax.annotate("Pierce design goal: 0.33\"", xy=(300, 0.33),
            textcoords="offset points", xytext=(0, 8), fontsize=10, color="gray")

ax.set_xlabel("Aperture Diameter (cm)")
ax.set_ylabel("Angular Resolution (arcsec)")
ax.set_title("Diffraction Limit vs. Seeing: Why Aperture Alone Is Not Enough\n"
             "회절 한계 vs. Seeing: 구경만으로는 부족한 이유")
ax.set_ylim(0, 3)
ax.set_xlim(0, 420)
ax.legend(loc="upper right")

# Shade the seeing-limited regime
ax.fill_between(apertures_cm, 1.5, 3, alpha=0.05, color="orange",
                label="Seeing-limited regime")
ax.annotate("← Seeing-limited regime\n   (구경 키워도 분해능 향상 없음)",
            xy=(250, 2.2), fontsize=11, color="darkorange",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

# Print key numbers
print("\n" + "=" * 50)
print("McMath: diffraction limit vs actual performance")
print("=" * 50)
dl_mcmath = diffraction_limit_arcsec(550, 160)
print(f"Diffraction limit (550 nm): {dl_mcmath:.3f} arcsec")
print(f"Design goal:                0.33 arcsec")
print(f"Typical daytime seeing:     1.5 arcsec")
print(f"Best nighttime star image:  0.2 arcsec")
print(f"Ratio seeing/diffraction:   {1.5/dl_mcmath:.1f}x")

## Part 3: 분광기 설계의 역방향 논리 / Backward Logic of Spectrograph Design

Pierce는 분광기 설계가 **과학 목표에서 역산**된다고 설명합니다. 이 연쇄적 논리를 시각화합니다:

Pierce explains that spectrograph design is **derived backward from the science goal**:

$$\text{Science goal} \rightarrow R \rightarrow \text{grating size} \rightarrow f_{\text{collimator}} \rightarrow f/\# \rightarrow \text{telescope aperture}$$

In [ ]:
# Pierce's backward design chain — reproduce the logic

print("=" * 70)
print("McMath Spectrograph Design Chain (Pierce's backward logic)")
print("=" * 70)

# Step 1: Science goal -> plate resolution
print("\n--- Step 1: Science Goal ---")
print("Goal: precision photometry of Fraunhofer line profiles")
print("Problem: plate resolution 10 μm → grain noise too high for photometry")
print("Solution: use wider projected slit → coarser resolution on plate")
plate_res = 30  # μm
print(f"→ Selected plate resolution: {plate_res} μm")

# Step 2: Plate resolution -> f-ratio
print("\n--- Step 2: Plate Resolution → f-ratio ---")
f_ratio = 60
print(f"f/# = {f_ratio} → resolution {plate_res} μm at the plate")

# Step 3: f-ratio + grating size -> spectrograph focal length
print("\n--- Step 3: f-ratio + Grating → Spectrograph Focal Length ---")
grating_w = 150  # mm, width of grating
grating_h = 250  # mm, height
f_spec = grating_h * f_ratio / 1000  # approximate, in meters
print(f"Grating ruled area: {grating_w} × {grating_h} mm")
print(f"Spectrograph focal length needed: ~{f_spec:.0f} m")
print(f"(to match grating to f/{f_ratio})")

# Step 4: Overfill the grating -> collimator focal length
print("\n--- Step 4: Overfill Grating → Collimator Focal Length ---")
f_coll = 13.7  # meters
print(f"Chose to overfill grating → collimator focal length: {f_coll} m")

# Step 5: Collimator f-ratio -> telescope f-ratio
print("\n--- Step 5: Collimator → Telescope f-ratio ---")
print(f"Spectrograph collimator determines telescope f/# = {f_ratio}")

# Step 6: f-ratio + science -> aperture
print("\n--- Step 6: f-ratio + Aperture → Focal Length ---")
aperture_cm = 160
focal_length = aperture_cm * f_ratio / 100  # meters
print(f"Available quartz blanks: {aperture_cm} cm diameter")
print(f"At f/{f_ratio}: focal length = {aperture_cm} cm × {f_ratio} = {focal_length:.0f} m")
print(f"(Actual McMath: ~90 m)")

# Visualize the chain as a flowchart
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis("off")
ax.set_title("Backward Design Chain / 역방향 설계 연쇄", fontsize=14, fontweight="bold")

boxes = [
    (1, 1.5, "Science Goal\nLine profiles\n@ 0.01%"),
    (3, 1.5, f"Plate Res.\n{plate_res} μm\n→ f/{f_ratio}"),
    (5, 1.5, f"Grating\n{grating_w}×{grating_h}mm\n610 gr/mm"),
    (7, 1.5, f"Collimator\nf = {f_coll} m"),
    (9, 1.5, f"Telescope\nf/# = {f_ratio}"),
    (11.5, 1.5, f"McMath\nD=160cm\nf=90m"),
]
colors = ["#FFD700", "#FFA500", "#FF6347", "#4169E1", "#32CD32", "#9370DB"]

for i, (x, y, text) in enumerate(boxes):
    rect = patches.FancyBboxPatch((x - 0.8, y - 0.6), 1.6, 1.2,
                                   boxstyle="round,pad=0.1",
                                   facecolor=colors[i], alpha=0.3,
                                   edgecolor="black", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=9, fontweight="bold")

    if i < len(boxes) - 1:
        next_x = boxes[i + 1][0]
        ax.annotate("", xy=(next_x - 0.85, y), xytext=(x + 0.85, y),
                    arrowprops=dict(arrowstyle="->", color="black", lw=2))

ax.text(7, 0.3, "← Direction of design logic (역방향) ←", ha="center",
        fontsize=12, style="italic", color="gray")
ax.text(7, 2.6, "→ Direction of light (순방향) →", ha="center",
        fontsize=12, style="italic", color="blue")

plt.tight_layout()
plt.show()

## Part 4: 격자 분해능과 분산 / Grating Resolving Power and Dispersion

격자 방정식과 분해능:

Grating equation and resolving power:

$$m\lambda = d(\sin\alpha + \sin\beta)$$
$$R = mN$$

McMath의 610 gr/mm 격자 ($N = 155{,}000$)에서 다양한 차수(order)에 따른 분해능과 선형 분산을 계산합니다.

Calculate resolving power and linear dispersion for McMath's 610 gr/mm grating ($N = 155{,}000$) at various orders.

In [ ]:
# McMath grating parameters
grooves_per_mm = 610
ruled_width_mm = 250  # mm (along dispersion direction)
ruled_height_mm = 150  # mm
N_grooves = grooves_per_mm * ruled_width_mm  # illuminated grooves
d_mm = 1.0 / grooves_per_mm  # groove spacing in mm
f_camera = 13.7  # m, camera/collimator focal length

print("=" * 60)
print("McMath Grating Specifications")
print("=" * 60)
print(f"Groove density:   {grooves_per_mm} grooves/mm")
print(f"Ruled area:       {ruled_width_mm} × {ruled_height_mm} mm")
print(f"N (grooves):      {N_grooves:,}")
print(f"Groove spacing d: {d_mm*1000:.4f} μm = {d_mm:.6f} mm")
print(f"Camera focal len: {f_camera} m")

# Resolving power at various orders
print(f"\n{'Order m':>8} {'R (theory)':>12} {'R (double)':>12} {'Δλ at 5000Å':>14} {'Lin. Disp.':>14}")
print(f"{'':>8} {'= mN':>12} {'= 2mN':>12} {'(mÅ)':>14} {'(mm/Å)':>14}")
print("-" * 64)

for m in range(1, 8):
    R_single = m * N_grooves
    R_double = 2 * m * N_grooves
    delta_lambda = 5000.0 / R_single  # in Å (at λ=5000Å)
    # Linear dispersion: dλ/dx = d·cos(β) / (m·f_camera)
    # At near-Littrow: cos(β) ≈ cos(α), approximate
    # Pierce reports ~7.5 mm/Å at 5th order
    lin_disp = m * f_camera * 1000 / (d_mm * 1e7)  # approximate mm/Å
    print(f"{m:>8d} {R_single:>12,} {R_double:>12,} {delta_lambda*1000:>14.2f} {lin_disp:>14.2f}")

print(f"\nPierce measured: ~7.5 mm/Å at 5th order (linear dispersion)")
print(f"Pierce measured: R ≈ 500,000 (single, photo), 600,000 (double, photo)")
print(f"Pierce measured: R → 775,000 / 1,550,000 (photoelectric, Rayleigh)")

# Visualize resolving power vs order for single and double pass
orders = np.arange(1, 8)
R_single = orders * N_grooves
R_double = 2 * orders * N_grooves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: resolving power
ax1.bar(orders - 0.2, R_single / 1e6, 0.35, label="Single pass", color="steelblue", alpha=0.8)
ax1.bar(orders + 0.2, R_double / 1e6, 0.35, label="Double pass", color="coral", alpha=0.8)

ax1.axhline(y=0.5, color="green", linestyle="--", alpha=0.7, label="Pierce measured (single photo): 500k")
ax1.axhline(y=0.6, color="red", linestyle="--", alpha=0.7, label="Pierce measured (double photo): 600k")

ax1.set_xlabel("Diffraction Order (m)")
ax1.set_ylabel("Resolving Power R (×10⁶)")
ax1.set_title("Theoretical Resolving Power R = mN\n이론적 분해능")
ax1.legend(fontsize=9)
ax1.set_xticks(orders)

# Right: minimum resolvable wavelength difference at λ=5000Å
delta_lambda_single = 5000.0 / R_single * 1000  # mÅ
delta_lambda_double = 5000.0 / R_double * 1000  # mÅ

ax2.plot(orders, delta_lambda_single, "s-", color="steelblue", markersize=8,
         linewidth=2, label="Single pass Δλ")
ax2.plot(orders, delta_lambda_double, "o-", color="coral", markersize=8,
         linewidth=2, label="Double pass Δλ")

# Mark I₂ line separations from the paper
ax2.axhline(y=9.0, color="purple", linestyle=":", alpha=0.5)
ax2.annotate("I₂ line separation a=9.0 mÅ\n(Pierce measured at 5th order double pass)",
             xy=(5, 9.0), textcoords="offset points", xytext=(30, 15),
             fontsize=9, color="purple",
             arrowprops=dict(arrowstyle="->", color="purple"))

ax2.set_xlabel("Diffraction Order (m)")
ax2.set_ylabel("Δλ at λ=5000 Å (mÅ)")
ax2.set_title("Minimum Resolvable Δλ at λ=5000 Å\n최소 분해 파장차")
ax2.legend()
ax2.set_xticks(orders)

plt.tight_layout()
plt.show()

## Part 5: 열 변형 계수 비교 — Quartz vs. Metal / Thermal Distortion Factor: Quartz vs. Metal

Couder의 열 변형 계수:

Couder's thermal distortion factor:

$$K = \frac{\alpha \delta c}{m}$$

여기서 $\alpha$ = 열팽창 계수, $\delta$ = 밀도, $c$ = 비열, $m$ = 열전도도. Pierce는 $\alpha$와 $m$의 역상관 때문에 quartz와 metal이 놀랍도록 유사한 $K$ 값을 갖는다고 지적합니다.

where $\alpha$ = thermal expansion, $\delta$ = density, $c$ = specific heat, $m$ = thermal conductivity. Pierce notes that the inverse correlation between $\alpha$ and $m$ gives quartz and metals surprisingly similar $K$ values.

In [ ]:
# Material properties (CGS units)
# α: thermal expansion coefficient (1/°C)
# δ: density (g/cm³)
# c: specific heat (cal/g·°C)
# m: thermal conductivity (cal/cm·s·°C)
materials = {
    "Glass": {
        "alpha": 9.0e-6, "delta": 2.5, "c": 0.2, "m": 0.0025,
        "K_paper": 152e-4, "color": "#FF6347"
    },
    "Pyrex": {
        "alpha": 3.2e-6, "delta": 2.23, "c": 0.2, "m": 0.0027,
        "K_paper": 50e-4, "color": "#FFA500"
    },
    "Fused Quartz": {
        "alpha": 0.55e-6, "delta": 2.2, "c": 0.17, "m": 0.0033,
        "K_paper": 5e-4, "color": "#4169E1"
    },
    "Aluminum\n(356-T6)": {
        "alpha": 23.0e-6, "delta": 2.68, "c": 0.214, "m": 0.37,
        "K_paper": 6e-4, "color": "#32CD32"  # approximate
    },
    "Iron": {
        "alpha": 12.0e-6, "delta": 7.87, "c": 0.11, "m": 0.18,
        "K_paper": 6e-4, "color": "#808080"
    },
}

print("=" * 75)
print("Couder Thermal Distortion Factor K = αδc/m")
print("=" * 75)
print(f"{'Material':<16} {'α (×10⁻⁶)':>10} {'δ (g/cm³)':>10} {'c (cal/g°C)':>12} "
      f"{'m (cal/cm·s°C)':>15} {'K (×10⁻⁴)':>10} {'K (paper)':>10}")
print("-" * 85)

K_computed = {}
for name, props in materials.items():
    K = props["alpha"] * props["delta"] * props["c"] / props["m"]
    K_computed[name] = K
    name_clean = name.replace("\n", " ")
    print(f"{name_clean:<16} {props['alpha']*1e6:>10.2f} {props['delta']:>10.2f} "
          f"{props['c']:>12.3f} {props['m']:>15.4f} {K*1e4:>10.1f} {props['K_paper']*1e4:>10.0f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: bar chart of K values
names = list(materials.keys())
K_vals = [K_computed[n] * 1e4 for n in names]
K_paper = [materials[n]["K_paper"] * 1e4 for n in names]
colors = [materials[n]["color"] for n in names]

x = np.arange(len(names))
ax1.bar(x, K_vals, color=colors, alpha=0.7, edgecolor="black")
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace("\n", "\n") for n in names], fontsize=10)
ax1.set_ylabel("K (× 10⁻⁴ CGS)")
ax1.set_title("Couder Distortion Factor K\n(lower = less thermal distortion)")
ax1.set_yscale("log")

# Annotate the surprise
ax1.annotate("Quartz ≈ Metal!\n(counterintuitive)",
             xy=(3, K_vals[3]), xytext=(3.5, 50),
             fontsize=11, color="red", fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="red"),
             bbox=dict(boxstyle="round", facecolor="lightyellow"))

# Right: scatter plot showing α vs m tradeoff
fig2_alpha = [materials[n]["alpha"] * 1e6 for n in names]
fig2_m = [materials[n]["m"] for n in names]

ax2.scatter(fig2_alpha, fig2_m, c=colors, s=200, edgecolors="black", zorder=5)
for i, name in enumerate(names):
    name_clean = name.replace("\n", " ")
    ax2.annotate(name_clean, (fig2_alpha[i], fig2_m[i]),
                 textcoords="offset points", xytext=(8, 8), fontsize=10)

ax2.set_xlabel("Thermal Expansion α (× 10⁻⁶ /°C)")
ax2.set_ylabel("Thermal Conductivity m (cal/cm·s·°C)")
ax2.set_title("The α–m Tradeoff\n열팽창 vs 열전도도 상충 관계")
ax2.set_xscale("log")
ax2.set_yscale("log")

# Draw iso-K lines (K = αδc/m → m = αδc/K → for fixed K, m ∝ α)
alpha_range = np.logspace(-1, 2, 100)
for K_target, label in [(5e-4, "K=5"), (50e-4, "K=50"), (150e-4, "K=150")]:
    # Approximate: assume δc ≈ 0.5 (typical)
    m_iso = alpha_range * 1e-6 * 0.5 / K_target
    ax2.plot(alpha_range, m_iso, "--", alpha=0.3, color="gray")
    valid = (m_iso > 1e-4) & (m_iso < 1)
    if valid.any():
        idx = np.where(valid)[0][-1]
        ax2.annotate(f"{label}×10⁻⁴", (alpha_range[idx], m_iso[idx]),
                     fontsize=8, color="gray")

plt.tight_layout()
plt.show()

# Thermal equilibrium time for mirror cooling
print("\n" + "=" * 60)
print("Mirror Thermal Flatness Requirement")
print("=" * 60)
# For a 150cm × 20cm quartz mirror to stay flat within 1λ:
alpha_q = 0.55e-6  # 1/°C
D = 150  # cm diameter
t = 20   # cm thickness
wavelength = 550e-7  # cm (550 nm)
# Curvature from ΔT: R_curv = t / (α × ΔT)
# For flatness within 1λ over diameter D:
# sag = D² / (8 × R_curv) < λ
# → ΔT < 8 × λ × t / (α × D²)
delta_T_max = 8 * wavelength * t / (alpha_q * D**2)
print(f"Quartz mirror: {D} cm diam × {t} cm thick")
print(f"Max ΔT for λ-flatness: {delta_T_max:.2f} °C")
print(f"Pierce states: < 0.06 °C (consistent with our calculation)")

## Part 6: Double-Pass 산란광 감소 효과 / Double-Pass Scattered Light Reduction

Double-pass에서 산란광은 이론적으로 single-pass의 제곱으로 줄어듭니다:

In double-pass, scattered light theoretically reduces as the square of single-pass:

$$S_{\text{double}} \approx S_{\text{single}}^2$$

Pierce 측정: single pass ghost 5% + general scatter 8% → double pass 총 3%. 이것이 Fraunhofer 선 core intensity의 정밀 측정을 가능케 합니다.

Pierce measured: single-pass ghost 5% + general scatter 8% → double-pass total 3%.

In [ ]:
# Model scattered light in single vs double pass

# Simple model: a Fraunhofer absorption line with scattered light contamination
# True line profile: Voigt-like (approximate as Gaussian core + Lorentzian wings)
wavelength = np.linspace(-200, 200, 1000)  # mÅ from line center

# True absorption line (deep, narrow)
true_depth = 0.85  # central depth (85% absorbed)
sigma = 30  # mÅ, Gaussian width
true_profile = 1.0 - true_depth * np.exp(-wavelength**2 / (2 * sigma**2))

# Single-pass scattered light
scatter_single = 0.08  # 8% general + ghost contribution
observed_single = true_profile * (1 - scatter_single) + scatter_single

# Double-pass scattered light
scatter_double = 0.03  # 3% total in double pass
observed_double = true_profile * (1 - scatter_double) + scatter_double

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: line profiles
ax1.plot(wavelength, true_profile, "k-", linewidth=2, label="True profile")
ax1.plot(wavelength, observed_single, "r-", linewidth=1.5, alpha=0.8,
         label=f"Single pass (scatter={scatter_single*100:.0f}%)")
ax1.plot(wavelength, observed_double, "b-", linewidth=1.5, alpha=0.8,
         label=f"Double pass (scatter={scatter_double*100:.0f}%)")

# Show the error at line center
true_center = true_profile[500]
single_center = observed_single[500]
double_center = observed_double[500]
ax1.annotate(f"Error at line center:\nSingle: {(single_center-true_center)/true_center*100:.1f}%\n"
             f"Double: {(double_center-true_center)/true_center*100:.1f}%",
             xy=(0, true_center), xytext=(80, 0.3), fontsize=10,
             bbox=dict(boxstyle="round", facecolor="lightyellow"),
             arrowprops=dict(arrowstyle="->"))

ax1.set_xlabel("Δλ from line center (mÅ)")
ax1.set_ylabel("Normalized Intensity")
ax1.set_title("Effect of Scattered Light on Line Profile\n산란광이 선 프로파일에 미치는 영향")
ax1.legend()
ax1.set_ylim(0, 1.1)

# Right: scattered light reduction — theoretical vs measured
scatter_levels = np.linspace(0.01, 0.20, 100)
scatter_double_theory = scatter_levels ** 2

ax2.plot(scatter_levels * 100, scatter_double_theory * 100, "b-", linewidth=2,
         label="Theory: $S_{double} = S_{single}^2$")

# Pierce's measured points
ax2.plot(8, 3, "r*", markersize=15, zorder=5, label="Pierce measured (ghost+scatter)")
ax2.annotate("Pierce: 8% → 3%\n(higher than S² = 0.64%\ndue to non-ideal baffling)",
             xy=(8, 3), textcoords="offset points", xytext=(20, 15), fontsize=10,
             color="red", arrowprops=dict(arrowstyle="->", color="red"))

# Show theoretical expectation
ax2.plot(8, 0.64, "g^", markersize=12, zorder=5, label="Theoretical: 8%² = 0.64%")

ax2.set_xlabel("Single-Pass Scattered Light (%)")
ax2.set_ylabel("Double-Pass Scattered Light (%)")
ax2.set_title("Double-Pass Scattered Light Reduction\nDouble-pass 산란광 감소")
ax2.legend(loc="upper left")
ax2.set_xlim(0, 20)
ax2.set_ylim(0, 5)

plt.tight_layout()
plt.show()

# Summary table
print("=" * 50)
print("Scattered Light Summary (from Pierce)")
print("=" * 50)
print(f"Single pass — ghost structures: ~5%")
print(f"Single pass — general scatter:  ~8%")
print(f"Double pass — total:            ~3%")
print(f"Theoretical S²:                 {0.08**2*100:.2f}% (ideal)")
print(f"Actual/theoretical ratio:       {3/0.64:.1f}x")
print(f"→ Excess due to non-ideal baffling, but still")
print(f"   a major improvement for line-core measurements")

## Part 7: S/N Ratio — Image Slicer 유무에 따른 비교 / S/N with and without Image Slicer

Bowen image slicer의 효과를 분석합니다. Pierce는 image slicer 사용 시 S/N이 구경에 직접 비례($\propto A$)하고, 미사용 시 제곱근에 비례($\propto \sqrt{A}$)한다고 설명합니다.

Analyzing the effect of the Bowen image slicer. Pierce explains that with an image slicer, S/N scales directly with aperture ($\propto A$), versus $\propto \sqrt{A}$ without.

In [ ]:
# S/N scaling with aperture: with and without image slicer

apertures = np.linspace(10, 400, 100)  # cm

# Normalize to McMath aperture (160 cm)
A_ref = 160

# Without image slicer: S/N ∝ sqrt(A)
sn_no_slicer = np.sqrt(apertures / A_ref)

# With image slicer: S/N ∝ A
sn_with_slicer = apertures / A_ref

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(apertures, sn_no_slicer, "b-", linewidth=2.5,
        label="Without image slicer: S/N ∝ √A")
ax.plot(apertures, sn_with_slicer, "r-", linewidth=2.5,
        label="With image slicer: S/N ∝ A")

# Mark telescopes
tel_marks = {
    "Dunn\n(76 cm)": 76,
    "SST\n(100 cm)": 100,
    "McMath\n(160 cm)": 160,
    "DKIST\n(400 cm)": 400,
}
for name, apt in tel_marks.items():
    sn_no = np.sqrt(apt / A_ref)
    sn_yes = apt / A_ref
    ax.plot(apt, sn_no, "bo", markersize=8)
    ax.plot(apt, sn_yes, "rs", markersize=8)
    ax.annotate(name, (apt, sn_yes), textcoords="offset points",
                xytext=(8, 5), fontsize=10, fontweight="bold")

# Gain factor annotation
ax.annotate(f"DKIST gain over McMath:\n"
            f"  Without slicer: {np.sqrt(400/160):.1f}×\n"
            f"  With slicer: {400/160:.1f}×",
            xy=(350, 2.0), fontsize=11,
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

ax.axhline(y=1.0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Aperture Diameter (cm)")
ax.set_ylabel("Relative S/N (normalized to McMath = 1)")
ax.set_title("S/N Scaling with Aperture: Image Slicer Effect\n"
             "구경에 따른 S/N 스케일링: Image Slicer 효과")
ax.legend(fontsize=12)
ax.set_xlim(0, 420)

plt.tight_layout()
plt.show()

# Why this matters for spectroscopy
print("=" * 55)
print("Why Image Slicer Matters for Solar Spectroscopy")
print("=" * 55)
print(f"McMath (160 cm) vs hypothetical 30 cm telescope:")
print(f"  Without slicer: S/N gain = √(160/30) = {np.sqrt(160/30):.1f}×")
print(f"  With slicer:    S/N gain = 160/30 = {160/30:.1f}×")
print(f"\nThis explains why McMath needed to be so large —")
print(f"for 0.01% photometry, every factor matters.")

## Part 8: McMath 광학 경로 시각화 / McMath Optical Path Visualization

논문의 Fig. 2를 기반으로 McMath Solar Telescope의 광학 경로를 재구성합니다. 31° 57.5' 기울어진 경사면을 따라 빛이 이동하는 독특한 구조를 보여줍니다.

Reconstructing the optical path of the McMath Solar Telescope based on Fig. 2 of the paper, showing the distinctive inclined structure at 31° 57.5'.

In [ ]:
# McMath Solar Telescope — optical path schematic
fig, ax = plt.subplots(figsize=(16, 9))

# Telescope geometry
incline_angle = np.radians(32)  # 31° 57.5' ≈ 32°
above_ground = 60  # m
below_ground = 90  # m
total_path = above_ground + below_ground  # ~150 m

# Ground level at y=0
# Heliostat at top of tower (30m above ground level at top of incline)
tower_height = 30  # m
incline_length_above = above_ground / np.sin(incline_angle)
incline_length_below = below_ground / np.sin(incline_angle)

# Coordinate system: x = horizontal, y = vertical
# Heliostat position (top of tower at the high end of incline)
heliostat_x = 0
heliostat_y = above_ground + tower_height  # ~90m above bottom

# Ground level at heliostat base
ground_at_heliostat_y = above_ground  # top of incline is 60m above ground at obs room

# Ground level (flat for simplicity)
ground_y = 0

# Incline path: from heliostat down to the bottom
# The incline goes from heliostat area down at 32° below horizontal
# Actually, the light path goes: heliostat -> down the incline to No.2 mirror -> back up

# Draw the structure
# Tower
tower_base_x = 5
tower_top_x = tower_base_x
ax.plot([tower_base_x, tower_top_x], [0, tower_height + 5],
        "k-", linewidth=6, solid_capstyle="round")
ax.annotate("Heliostat\nTower\n(30m)", xy=(tower_base_x, tower_height / 2),
            fontsize=9, ha="center", color="brown",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

# Heliostat mirror at top
ax.plot([tower_base_x - 3, tower_base_x + 3], [tower_height + 5, tower_height + 5],
        "b-", linewidth=4)
ax.annotate("80\" Heliostat\n(No. 1 Mirror)", xy=(tower_base_x, tower_height + 7),
            fontsize=10, ha="center", color="blue", fontweight="bold")

# Sun rays coming in
for offset in [-2, 0, 2]:
    ax.annotate("", xy=(tower_base_x + offset, tower_height + 5),
                xytext=(tower_base_x + offset - 5, tower_height + 15),
                arrowprops=dict(arrowstyle="-|>", color="gold", lw=2))
ax.text(tower_base_x - 7, tower_height + 14, "Sunlight\n☀", fontsize=12,
        color="darkorange", fontweight="bold")

# Incline structure (above ground portion)
incline_end_x = tower_base_x + above_ground / np.tan(incline_angle)
incline_end_below_x = incline_end_x + below_ground / np.tan(incline_angle)

# Above ground enclosure
ax.fill_between(
    [tower_base_x + 3, incline_end_x],
    [tower_height - 2, -2],
    [tower_height + 2, 2],
    color="silver", alpha=0.3, label="Water-cooled enclosure"
)
ax.plot([tower_base_x + 3, incline_end_x], [tower_height, 0],
        "k--", linewidth=1, alpha=0.5)

# Below ground tunnel
ax.fill_between(
    [incline_end_x, incline_end_below_x],
    [-2, -below_ground * np.sin(incline_angle) / np.cos(incline_angle) * 0.3 - 2],
    [2, -below_ground * np.sin(incline_angle) / np.cos(incline_angle) * 0.3 + 2],
    color="saddlebrown", alpha=0.2, label="Underground tunnel"
)

# Simplified light path
# Sunlight → heliostat → down to No.2 mirror → back up to No.3 → observation room

# No.2 mirror (60-inch primary, at the bottom)
no2_x = incline_end_x + 60 / np.tan(incline_angle)
no2_y = -30
ax.plot([no2_x - 2, no2_x + 2], [no2_y - 1, no2_y + 1], "r-", linewidth=4)
ax.annotate("60\" Primary\n(No. 2 Mirror)\nf = 90m",
            xy=(no2_x, no2_y - 4), fontsize=10, ha="center", color="red",
            fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="mistyrose", alpha=0.8))

# No.3 mirror (flat, directs to observation room)
no3_x = incline_end_x + 20
no3_y = -8
ax.plot([no3_x - 1.5, no3_x + 1.5], [no3_y - 0.5, no3_y + 0.5], "g-", linewidth=4)
ax.annotate("48\" Flat\n(No. 3 Mirror)",
            xy=(no3_x, no3_y + 3), fontsize=10, ha="center", color="green",
            fontweight="bold")

# Observation room
obs_x = incline_end_x + 5
obs_y = -12
obs_room = patches.FancyBboxPatch((obs_x - 8, obs_y - 5), 16, 10,
                                   boxstyle="round,pad=0.5",
                                   facecolor="lightyellow", edgecolor="black",
                                   linewidth=2, alpha=0.7)
ax.add_patch(obs_room)
ax.text(obs_x, obs_y, "Observation Room\n관측실\n(Spectrographs)", fontsize=10,
        ha="center", va="center", fontweight="bold")

# Light path arrows (simplified)
# Sunlight to heliostat
# Heliostat to No.2 (down the incline)
ax.annotate("", xy=(no2_x, no2_y),
            xytext=(tower_base_x, tower_height + 3),
            arrowprops=dict(arrowstyle="-|>", color="gold", lw=2.5,
                          connectionstyle="arc3,rad=0"))
ax.text(tower_base_x + 30, tower_height * 0.3, "Incoming light\n(down incline)",
        fontsize=9, color="goldenrod", rotation=-25)

# No.2 to No.3 (reflected back up)
ax.annotate("", xy=(no3_x, no3_y),
            xytext=(no2_x, no2_y),
            arrowprops=dict(arrowstyle="-|>", color="orange", lw=2.5))
ax.text(no2_x - 5, no2_y + 8, "Reflected\n(back up)", fontsize=9, color="darkorange")

# No.3 to observation room
ax.annotate("", xy=(obs_x, obs_y + 3),
            xytext=(no3_x, no3_y),
            arrowprops=dict(arrowstyle="-|>", color="red", lw=2.5))

# Ground level line
ax.axhline(y=0, color="brown", linewidth=3, linestyle="-")
ax.text(incline_end_x + 30, 1, "Ground Level / 지면", fontsize=10, color="brown")

# Spectrograph tank annotation
ax.annotate("Vacuum Spectrograph\n(21m tank, 17 tons\nrotates on oil film)",
            xy=(obs_x, obs_y - 8), fontsize=9, ha="center", color="purple",
            bbox=dict(boxstyle="round", facecolor="lavender", alpha=0.8))

# Scale and labels
ax.set_xlim(-15, incline_end_x + 50)
ax.set_ylim(-40, tower_height + 20)
ax.set_aspect("equal")
ax.set_xlabel("Horizontal Distance (m, approximate)")
ax.set_ylabel("Elevation (m, approximate)")
ax.set_title("McMath Solar Telescope — Optical Path Schematic\n"
             "McMath 태양 망원경 — 광학 경로 모식도 (after Pierce 1964, Fig. 2)",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right")

# Key dimensions annotation
ax.annotate(f"Incline angle: 32°\nAbove ground: ~60 m\nBelow ground: ~90 m\n"
            f"Total optical path: ~155 m",
            xy=(incline_end_x + 25, tower_height + 5), fontsize=10,
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

plt.tight_layout()
plt.show()

## 요약 / Summary

| Part | 주제 / Topic | 핵심 결과 / Key Result |
|------|------------|---------------------|
| 1 | Image Scale | McMath: 0.436 mm/arcsec → 84 cm solar image ("~1 yard") |
| 2 | Diffraction vs Seeing | Diffraction limit 0.084" but seeing ~1.5" → 18× gap |
| 3 | Design Chain | Science goal → plate resolution → grating → collimator → telescope |
| 4 | Grating Resolution | R = 600,000 (double pass photo), up to 1,550,000 (photoelectric) |
| 5 | Thermal Distortion | Quartz K ≈ Metal K (counterintuitive: α and m inversely correlated) |
| 6 | Scattered Light | Double pass: 8% → 3% (critical for line-core photometry) |
| 7 | S/N Scaling | With image slicer: S/N ∝ A (not √A) — justifies large aperture |
| 8 | Optical Path | 32° incline, 155m path, heliostat on 30m tower, water-cooled enclosure |

**다음 논문 / Next Paper**: #2 Dunn (1969) — Vacuum Tower Telescope
→ Pierce가 해결하지 못한 internal seeing 문제를 진공 광로로 근본적으로 해결하는 접근법
→ Fundamentally solves internal seeing via an evacuated optical path